# Essay Corpus Diversity Analysis

> **Data sensitivity:** This notebook uses restricted admissions data. Run locally only and do not commit data files or outputs to the repository.

## Purpose

Measures how semantically similar essays are to each other within and across corpora.

**Why this matters:**
- **Low diversity (high average similarity)** is a strong signal that essays were generated by a language model using a small set of templates or prompts. Real applicant essays, even on the same topic, tend to vary considerably in voice, structure, and specific detail.
- **Near-duplicate pairs** (cosine similarity > 0.95) almost certainly share a common source â€” either copy-paste, paraphrasing, or the same LLM prompt.
- **Cross-corpus comparison** lets you directly contrast real vs. synthetic distributions to calibrate what "suspicious" looks like.

## Workflow
1. Load one or more essay datasets
2. Embed with `all-MiniLM-L6-v2` (same model as topic modeling, for consistency)
3. Compute pairwise cosine similarity (sampled for large corpora)
4. Plot similarity distributions per corpus
5. Identify near-duplicate pairs
6. Summary stats table

## Install dependencies (if needed)

In [ ]:
# Run once if packages are missing
#!pip install sentence-transformers pandas matplotlib seaborn scikit-learn openpyxl

## Imports

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')

## Configuration

In [ ]:
# ── Which corpora to analyse ──────────────────────────────────────────────────
# Add or remove entries to control what gets loaded and compared.
CORPORA = {
    'real':      Path('../data/real/essays.xlsx'),
    'synthetic': Path('../data/synthetic/synthetic_essays.csv'),
    'freeform':  Path('../data/freeform/freeform_essays.csv'),
}

# Embedding model — same as topic_modeling.ipynb for consistency.
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'

# For large corpora (e.g. real ~15k), computing all N² pairs is slow and
# memory-intensive. We randomly sample this many essays instead.
# Set to None to use the full corpus (fine for <2k essays).
SAMPLE_N = None

# Cosine similarity threshold above which two essays are flagged as near-duplicates.
NEAR_DUPLICATE_THRESHOLD = 0.9

RANDOM_STATE = 42
OUTPUT_DIR = Path('../outputs/diversity')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 1: Load corpora

In [ ]:
def load_corpus(name, path, sample_n=SAMPLE_N):
    p = Path(path)
    if not p.exists():
        print(f'  [{name}] File not found: {p} â€” skipping.')
        return None
    if p.suffix.lower() in ('.xlsx', '.xls'):
        df = pd.read_excel(p)
        df['essay'] = df['Essay Response']
        if 'Essay Score' in df.columns:
            df['score'] = df['Essay Score']
            df = df.dropna(subset=["score", "essay"])
    else:
        df = pd.read_csv(p)
        if 'essay' not in df.columns:
            raise ValueError(f'{p.name} has no "essay" column. Columns: {list(df.columns)}')
        if 'score' not in df.columns and 'proxy_score' in df.columns:
            df['score'] = df['proxy_score']
    df = df.dropna(subset=['essay'])
    df['corpus'] = name
    if sample_n and len(df) > sample_n:
        df = df.sample(sample_n, random_state=RANDOM_STATE).reset_index(drop=True)
        print(f'  [{name}] Sampled {sample_n:,} / {len(df):,} essays')
    else:
        print(f'  [{name}] Loaded {len(df):,} essays')
    return df


corpora = {}
for name, path in CORPORA.items():
    df = load_corpus(name, path)
    if df is not None:
        corpora[name] = df

print(f'\nLoaded {len(corpora)} corpus/corpora: {list(corpora.keys())}')

## Step 2: Embed essays

In [ ]:
model = SentenceTransformer(EMBED_MODEL, device='cuda')

embeddings = {}
for name, df in corpora.items():
    print(f'Embedding [{name}] ({len(df):,} essays)â€¦')
    emb = model.encode(
        df['essay'].tolist(),
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,   # unit vectors â†’ dot product == cosine similarity
    )
    embeddings[name] = emb
    print(f'  Shape: {emb.shape}')

print('\nEmbedding complete.')

## Step 3: Compute pairwise cosine similarities

We exclude self-similarity (diagonal) and keep only the upper triangle to avoid double-counting pairs.

In [ ]:
sim_scores = {}   # name â†’ 1-D array of upper-triangle similarity values

for name, emb in embeddings.items():
    print(f'Computing similarity matrix for [{name}] ({len(emb):,} essays)â€¦')
    # embeddings are already L2-normalised, so dot product == cosine similarity
    sim_matrix = emb @ emb.T
    # Extract upper triangle (excluding diagonal)
    idx = np.triu_indices(len(emb), k=1)
    sims = sim_matrix[idx]
    sim_scores[name] = sims
    print(f'  {len(sims):,} pairs | mean={sims.mean():.4f} | median={np.median(sims):.4f}')

print('\nDone.')

## Step 4: Similarity distributions

In [ ]:
fig, axes = plt.subplots(1, len(sim_scores), figsize=(6 * len(sim_scores), 4), sharey=False)
if len(sim_scores) == 1:
    axes = [axes]

palette = sns.color_palette('tab10', len(sim_scores))

for ax, (name, sims), color in zip(axes, sim_scores.items(), palette):
    ax.hist(sims, bins=80, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(sims.mean(), color='black', linestyle='--', linewidth=1.2,
               label=f'mean={sims.mean():.3f}')
    ax.axvline(np.median(sims), color='grey', linestyle=':', linewidth=1.2,
               label=f'median={np.median(sims):.3f}')
    ax.set_title(f'{name} corpus')
    ax.set_xlabel('Cosine similarity')
    ax.set_ylabel('Pair count')
    ax.legend(fontsize=8)

plt.suptitle('Pairwise cosine similarity distribution per corpus', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'similarity_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to', OUTPUT_DIR / 'similarity_distributions.png')

### Overlay comparison (all corpora on one axis)

In [ ]:
if len(sim_scores) > 1:
    fig, ax = plt.subplots(figsize=(9, 4))
    for (name, sims), color in zip(sim_scores.items(), palette):
        ax.hist(sims, bins=80, alpha=0.5, color=color, label=name, edgecolor='none', density=True)
        ax.axvline(sims.mean(), color=color, linestyle='--', linewidth=1.5)
    ax.set_xlabel('Cosine similarity')
    ax.set_ylabel('Density')
    ax.set_title('Pairwise similarity distributions: all corpora overlaid')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'similarity_overlay.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved to', OUTPUT_DIR / 'similarity_overlay.png')

## Step 5: Summary statistics

In [ ]:
rows = []
for name, sims in sim_scores.items():
    n_essays = len(corpora[name])
    n_dups = (sims >= NEAR_DUPLICATE_THRESHOLD).sum()
    rows.append({
        'corpus':             name,
        'n_essays':           n_essays,
        'n_pairs':            len(sims),
        'mean_sim':           round(float(sims.mean()), 4),
        'median_sim':         round(float(np.median(sims)), 4),
        'p90_sim':            round(float(np.percentile(sims, 90)), 4),
        'p99_sim':            round(float(np.percentile(sims, 99)), 4),
        f'pairs: {NEAR_DUPLICATE_THRESHOLD}': int(n_dups),
        '% pairs near-dup':   round(100 * n_dups / len(sims), 2),
    })

stats_df = pd.DataFrame(rows).set_index('corpus')
stats_df.to_csv(OUTPUT_DIR / 'diversity_stats.csv')
print(stats_df.to_string())

## Step 6: Near-duplicate pairs

Essays with cosine similarity â‰¥ `NEAR_DUPLICATE_THRESHOLD` are likely paraphrases of each other or generated from the same prompt.

In [ ]:
for name, emb in embeddings.items():
    df = corpora[name].reset_index(drop=True)
    sim_matrix = emb @ emb.T
    rows_i, rows_j = np.where(
        (sim_matrix >= NEAR_DUPLICATE_THRESHOLD) & (np.triu(np.ones_like(sim_matrix, dtype=bool), k=1))
    )
    if len(rows_i) == 0:
        print(f'[{name}] No near-duplicate pairs found (threshold={NEAR_DUPLICATE_THRESHOLD})')
        continue

    dup_pairs = pd.DataFrame({
        'idx_a':   rows_i,
        'idx_b':   rows_j,
        'sim':     sim_matrix[rows_i, rows_j].round(4),
        'essay_a': df.loc[rows_i, 'essay'].str[:120].values,
        'essay_b': df.loc[rows_j, 'essay'].str[:120].values,
    }).sort_values('sim', ascending=False)

    out_path = OUTPUT_DIR / f'near_duplicates_{name}.csv'
    dup_pairs.to_csv(out_path, index=False)
    print(f'[{name}] {len(dup_pairs):,} near-duplicate pair(s): saved to {out_path}')
    display(dup_pairs.head(10))

## Step 7: Similarity heatmap (sample)

A heatmap of a random 100-essay sample. Bright clusters off-diagonal indicate groups of very similar essays.

In [ ]:
HEATMAP_N = 100   # number of essays to sample for the heatmap

for name, emb in embeddings.items():
    rng = np.random.default_rng(RANDOM_STATE)
    idx = rng.choice(len(emb), size=min(HEATMAP_N, len(emb)), replace=False)
    sub = emb[idx]
    sim_sub = sub @ sub.T

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(sim_sub, vmin=0, vmax=1, cmap='viridis', aspect='auto')
    plt.colorbar(im, ax=ax, label='Cosine similarity')
    ax.set_title(f'[{name}] Pairwise similarity: ” {len(idx)}-essay sample')
    ax.set_xlabel('Essay index (sample)')
    ax.set_ylabel('Essay index (sample)')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f'heatmap_{name}.png', dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved to {OUTPUT_DIR}/heatmap_{name}.png')